In [2]:
# Set project directory
%cd /content/drive/MyDrive/app_note_rag

/content/drive/MyDrive/app_note_rag


In [3]:
# Install embedding dependencies
!pip install -q sentence-transformers psycopg langchain-text-splitters

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 215.5/215.5 kB 5.5 MB/s eta 0:00:00


In [4]:
# Test BGE-M3 embedding generation
from src.rag.ingestion.embedder import embedder


text = "LangGraph is used to build stateful AI workflows."

embedding = embedder.embed_text(text)

print(f"Embedding dimension: {len(embedding)}")
print(f"First 5 values: {embedding[:5]}")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/15.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

Embedding dimension: 1024
First 5 values: [-0.05850336328148842, -0.003621085546910763, -0.005224073771387339, -0.03424195200204849, -0.028105780482292175]


In [5]:
# Test text chunking

from src.rag.ingestion.chunker import Chunker


text = """
LangGraph is a framework for building stateful AI applications.
It allows developers to define workflows as graphs with nodes and edges.
Nodes can perform different operations such as calling an LLM or executing tools.
The graph maintains state throughout the workflow.
LangGraph can also support human-in-the-loop workflows and persistence.
""" * 10


chunker = Chunker(
    chunk_size=1000,
    chunk_overlap=200,
)

chunks = chunker.split_text(text)

print(f"Number of chunks: {len(chunks)}")

for index, chunk in enumerate(chunks):
    print(f"\nChunk {index}:")
    print(chunk[:200])

Number of chunks: 5

Chunk 0:
LangGraph is a framework for building stateful AI applications.
It allows developers to define workflows as graphs with nodes and edges.
Nodes can perform different operations such as calling an LLM o

Chunk 1:
LangGraph is a framework for building stateful AI applications.
It allows developers to define workflows as graphs with nodes and edges.
Nodes can perform different operations such as calling an LLM o

Chunk 2:
LangGraph is a framework for building stateful AI applications.
It allows developers to define workflows as graphs with nodes and edges.
Nodes can perform different operations such as calling an LLM o

Chunk 3:
LangGraph is a framework for building stateful AI applications.
It allows developers to define workflows as graphs with nodes and edges.
Nodes can perform different operations such as calling an LLM o

Chunk 4:
LangGraph is a framework for building stateful AI applications.
It allows developers to define workflows as graphs with nodes 

In [6]:
# Test note ingestion

from src.rag.ingestion.service import ingestion_service


text = """
LangGraph is a framework for building stateful AI applications.
It allows developers to define workflows as graphs with nodes and edges.
Nodes can call language models, execute tools, and transform state.
The graph can persist state across multiple interactions.
"""


result = ingestion_service.process_note(text)

print(f"Number of chunks: {len(result)}")
print(f"First chunk index: {result[0]['chunk_index']}")
print(f"First chunk length: {len(result[0]['content'])}")
print(f"First embedding dimension: {len(result[0]['embedding'])}")

Number of chunks: 1
First chunk index: 0
First chunk length: 262
First embedding dimension: 1024


In [7]:
# Enable pgvector for the current database

!sudo -u postgres psql app_note_rag -c "CREATE EXTENSION IF NOT EXISTS vector;"

sudo: unknown user postgres
sudo: error initializing audit plugin sudoers_audit


In [8]:
# Install pgvector for PostgreSQL

!apt-get update -qq
!apt-get install -y -qq postgresql-16-pgvector

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Preconfiguring packages ...
Selecting previously unselected package libjson-perl.
(Reading database ... 122809 files and directories currently installed.)
Preparing to unpack .../00-libjson-perl_4.10000-1_all.deb ...
Unpacking libjson-perl (4.10000-1) ...
Selecting previously unselected package postgresql-client-common.
Preparing to unpack .../01-postgresql-client-common_257build1.1_all.deb ...
Unpacking postgresql-client-common (257build1.1) ...
Selecting previously unselected package ssl-cert.
Preparing to unpack .../02-ssl-cert_1.1.2ubuntu1_all.deb ...
Unpacking ssl-cert (1.1.2ubuntu1) ...
Selecting previously unselected package postgresql-common.
Preparing to unpack .../03-postgresql-common_257build1.1_all.deb ...
Adding 'diversion of /usr/bin/pg_config to /usr/bin/pg_config.libpq-dev by postgres

In [9]:
# Restart PostgreSQL after installing pgvector

!service postgresql restart

[ OK ]


In [10]:
# Enable pgvector for the current database

!sudo -u postgres psql app_note_rag -c "CREATE EXTENSION IF NOT EXISTS vector;"

psql: error: connection to server on socket "/var/run/postgresql/.s.PGSQL.5432" failed: FATAL:  database "app_note_rag" does not exist


In [11]:
# Initialize PostgreSQL for the current runtime

from src.database import init_database

init_database()

✅ PostgreSQL initialized and database restored.


In [12]:
# Create the current database schema

from src.schema import create_tables

create_tables()

print("Database schema is up to date.")

Database schema is up to date.


In [13]:
# Add embedding column to the existing chunks table

from src.database import get_connection

connection = get_connection()

connection.execute("""
    ALTER TABLE chunks
    ADD COLUMN IF NOT EXISTS embedding vector(1024)
""")

connection.commit()
connection.close()

print("Embedding column is ready.")

Embedding column is ready.


In [14]:
# Test chunk persistence

from src.rag.ingestion.service import ingestion_service
from src.rag.ingestion.repository import save_chunks

note_id = 2

note = """
Artificial intelligence research requires a combination of machine learning,
deep learning, natural language processing, and careful experimentation.
Researchers need reliable datasets, appropriate evaluation methods, and
reproducible experiments to validate their findings.
"""

chunks = ingestion_service.process_note(note)

save_chunks(
    note_id=note_id,
    chunks=chunks,
)

print(f"Saved {len(chunks)} chunks for note {note_id}.")

Saved 1 chunks for note 2.


In [15]:
# Verify stored chunk and embedding

from src.database import get_connection

connection = get_connection()

row = connection.execute("""
    SELECT
        id,
        note_id,
        chunk_index,
        LEFT(content, 100),
        vector_dims(embedding)
    FROM chunks
    WHERE note_id = 2
    ORDER BY chunk_index
    LIMIT 1
""").fetchone()

connection.close()

print(f"Chunk ID: {row[0]}")
print(f"Note ID: {row[1]}")
print(f"Chunk index: {row[2]}")
print(f"Content preview: {row[3]}")
print(f"Embedding dimension: {row[4]}")

Chunk ID: 2
Note ID: 2
Chunk index: 0
Content preview: Artificial intelligence research requires a combination of machine learning,
deep learning, natural 
Embedding dimension: 1024


In [16]:
# Backup the current PostgreSQL database

from src.database import backup_database

backup_database()

✅ Database backup saved to: /content/drive/MyDrive/app_note_rag/db_backups/app_note_rag_backup.sql


In [17]:
# Test semantic retrieval

from src.rag.retrieval.semantic import semantic_search

results = semantic_search(
    query="What is required for AI research?",
    user_id=4,
    top_k=5,
)

for result in results:
    print(f"Chunk ID: {result['chunk_id']}")
    print(f"Note ID: {result['note_id']}")
    print(f"Similarity: {result['similarity']:.4f}")
    print(f"Content: {result['content']}")
    print("-" * 80)

Chunk ID: 2
Note ID: 2
Similarity: 0.7084
Content: Artificial intelligence research requires a combination of machine learning,
deep learning, natural language processing, and careful experimentation.
Researchers need reliable datasets, appropriate evaluation methods, and
reproducible experiments to validate their findings.
--------------------------------------------------------------------------------
Chunk ID: 3
Note ID: 2
Similarity: 0.7084
Content: Artificial intelligence research requires a combination of machine learning,
deep learning, natural language processing, and careful experimentation.
Researchers need reliable datasets, appropriate evaluation methods, and
reproducible experiments to validate their findings.
--------------------------------------------------------------------------------


In [19]:
# Test keyword retrieval with an exact term

results = keyword_search(
    query="research",
    user_id=4,
    top_k=5,
)

for result in results:
    print(f"Chunk ID: {result['chunk_id']}")
    print(f"Score: {result['score']:.4f}")
    print(f"Content: {result['content']}")

Chunk ID: 2
Score: 0.0608
Content: Artificial intelligence research requires a combination of machine learning,
deep learning, natural language processing, and careful experimentation.
Researchers need reliable datasets, appropriate evaluation methods, and
reproducible experiments to validate their findings.
Chunk ID: 3
Score: 0.0608
Content: Artificial intelligence research requires a combination of machine learning,
deep learning, natural language processing, and careful experimentation.
Researchers need reliable datasets, appropriate evaluation methods, and
reproducible experiments to validate their findings.


In [20]:
# Test hybrid retrieval

from src.rag.retrieval.hybrid import hybrid_search

results = hybrid_search(
    query="artificial intelligence research",
    user_id=4,
    top_k=5,
)

for result in results:
    print(f"Chunk ID: {result['chunk_id']}")
    print(f"Note ID: {result['note_id']}")
    print(f"Semantic score: {result['semantic_score']:.4f}")
    print(f"Keyword score: {result['keyword_score']:.4f}")
    print(f"RRF score: {result['rrf_score']:.6f}")
    print(f"Content: {result['content']}")
    print("-" * 80)

Chunk ID: 2
Note ID: 2
Semantic score: 0.7106
Keyword score: 0.2683
RRF score: 0.032787
Content: Artificial intelligence research requires a combination of machine learning,
deep learning, natural language processing, and careful experimentation.
Researchers need reliable datasets, appropriate evaluation methods, and
reproducible experiments to validate their findings.
--------------------------------------------------------------------------------
Chunk ID: 3
Note ID: 2
Semantic score: 0.7106
Keyword score: 0.2683
RRF score: 0.032258
Content: Artificial intelligence research requires a combination of machine learning,
deep learning, natural language processing, and careful experimentation.
Researchers need reliable datasets, appropriate evaluation methods, and
reproducible experiments to validate their findings.
--------------------------------------------------------------------------------
